# yt2tg — YouTube → Telegram Uploader

**Resume-safe**: state is saved after each upload. If Colab disconnects, just re-run — already-uploaded videos are skipped automatically.

---
### First time only (run cells in order):
1. **Cell 1** — Clone repo & install deps
2. **Cell 2** — Fill in your Telegram credentials
3. **Cell 3** — Upload your `cookies.txt`
4. **Cell 4** — Preview what will be downloaded (optional)
5. **Cell 5** — Run the actual download & upload

### Resuming after Colab disconnects:
Re-run **Cell 1** → **Cell 2** → **Cell 5**. Already uploaded videos are skipped.

> **Cookies expired?** Re-run Cell 3 with a fresh cookies.txt.

In [ ]:
# ═══════════════════════════════════════════════
# Cell 1: Clone repo & install dependencies
# ═══════════════════════════════════════════════
import os

REPO_DIR = '/content/yt2tg'

# Clone on first run, pull updates on subsequent runs
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/sudo-change/yt-tg.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

# Install Python dependencies
# tgcrypto speeds up Pyrogram (works on Linux/Colab, optional)
!pip install -q pyrogram tgcrypto yt-dlp

# Confirm yt-dlp version
!yt-dlp --version
print('Setup complete!')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 2: Telegram credentials
# ═══════════════════════════════════════════════
# Fill in your values below, then run this cell.
# This writes config.json — it is gitignored and never uploaded.
import json, os

REPO_DIR = '/content/yt2tg'

# ──────────────────────────────────────────────────────────────
BOT_TOKEN     = ""    # @BotFather -> /newbot -> copy token
API_ID        = ""    # https://my.telegram.org -> App configuration
API_HASH      = ""    # https://my.telegram.org -> App configuration
GROUP_CHAT_ID = 0     # Your supergroup ID — negative number e.g. -1001234567890
              #   How to get it: add @userinfobot to your group, it replies with the ID
# ──────────────────────────────────────────────────────────────

config_path = os.path.join(REPO_DIR, 'config.json')
config = {
    'bot_token':        BOT_TOKEN,
    'api_id':           API_ID,
    'api_hash':         API_HASH,
    'group_chat_id':    GROUP_CHAT_ID,
    'channel_mappings': {}
}
# Preserve existing channel_mappings (topic IDs already created)
if os.path.exists(config_path):
    with open(config_path) as f:
        existing = json.load(f)
    config['channel_mappings'] = existing.get('channel_mappings', {})

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f'Config saved to {config_path}')
print(f'  Bot token : {BOT_TOKEN[:15]}...')
print(f'  API ID    : {API_ID}')
print(f'  Group ID  : {GROUP_CHAT_ID}')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 3: Upload cookies.txt
# ═══════════════════════════════════════════════
# How to get cookies.txt:
#   1. Install browser extension: "Get cookies.txt LOCALLY" (Chrome/Firefox)
#   2. Log into YouTube in your browser
#   3. Click the extension -> export for youtube.com
#   4. Upload that file below
#
# Re-run this cell whenever YouTube says cookies are expired.
from google.colab import files
import shutil, os

REPO_DIR = '/content/yt2tg'

uploaded = files.upload()
if uploaded:
    fname = list(uploaded.keys())[0]
    dest  = os.path.join(REPO_DIR, 'cookies.txt')
    shutil.move(fname, dest)
    print(f'Cookies saved -> {dest}')
else:
    print('No file uploaded. Membership/private videos will fail without cookies.')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 4: Preview (dry-run — no downloads)
# ═══════════════════════════════════════════════
# Lists videos that WOULD be processed. Nothing is downloaded or uploaded.
# Good for checking membership video counts before committing.

REPO_DIR = '/content/yt2tg'

URLS = [
    'https://youtube.com/@NahamSec',           # channel URL example
    # 'https://youtube.com/watch?v=abc123',    # single video example
]

# Flags:
#   -m   membership-only videos
#   ''   all videos (will ask confirmation before downloading)
FLAGS = '-m'

url_args = ' '.join(f'"{u}"' for u in URLS)
!cd {REPO_DIR} && python yt2tg.py --dry-run {FLAGS} {url_args}

In [ ]:
# ═══════════════════════════════════════════════
# Cell 5: Download & Upload  ← MAIN CELL
# ═══════════════════════════════════════════════
# If Colab disconnects mid-run:
#   -> Re-run Cell 1, Cell 2, then this cell again
#   -> Already uploaded videos are skipped (tracked in state.json)
#   -> Partial downloads resume automatically via yt-dlp --continue

REPO_DIR = '/content/yt2tg'

URLS = [
    'https://youtube.com/@NahamSec',           # channel: all membership videos
    # 'https://youtube.com/watch?v=abc123',    # single video
    # 'https://youtube.com/@AnotherChannel',   # add more channels
]

# Flags:
#   -m   membership-only  (omit to get all videos)
#   -y   skip confirmation prompts (keep this for Colab)
FLAGS = '-m -y'

url_args = ' '.join(f'"{u}"' for u in URLS)
!cd {REPO_DIR} && python yt2tg.py {FLAGS} {url_args}

In [ ]:
# ═══════════════════════════════════════════════
# Cell 6: Check progress
# ═══════════════════════════════════════════════
import json, os

REPO_DIR = '/content/yt2tg'
state_path = os.path.join(REPO_DIR, 'state.json')

if os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    completed = state.get('completed', [])
    failed    = state.get('failed', {})
    print(f'Completed : {len(completed)} videos uploaded')
    print(f'Failed    : {len(failed)} videos')
    if failed:
        print('\nFailed videos (re-run Cell 5 after fixing cookies to retry):')
        for vid_id, reason in failed.items():
            print(f'  {vid_id}: {reason[:100]}')
else:
    print('No state.json yet — Cell 5 has not been run')

In [ ]:
# ═══════════════════════════════════════════════
# Cell 7: Force re-upload specific videos
# ═══════════════════════════════════════════════
# If a video uploaded incorrectly and you want to redo it,
# add its video ID below and run this cell, then re-run Cell 5.
import json, os

REPO_DIR   = '/content/yt2tg'
state_path = os.path.join(REPO_DIR, 'state.json')

# Paste video IDs from YouTube URLs (the part after ?v=)
VIDEO_IDS_TO_RETRY = [
    # 'abc123xyz',
]

if VIDEO_IDS_TO_RETRY and os.path.exists(state_path):
    with open(state_path) as f:
        state = json.load(f)
    state['completed'] = [v for v in state.get('completed', []) if v not in VIDEO_IDS_TO_RETRY]
    for v in VIDEO_IDS_TO_RETRY:
        state.get('failed', {}).pop(v, None)
    with open(state_path, 'w') as f:
        json.dump(state, f, indent=2)
    print(f'Removed {len(VIDEO_IDS_TO_RETRY)} video(s) from state. Re-run Cell 5 to process them.')
else:
    print('Add video IDs to VIDEO_IDS_TO_RETRY list above first.')